# Translate Text — Azure AI Translator

This notebook translates a piece of text into multiple target languages in a single API call.

The service automatically detects the source language when no `from_language` is specified.

In [1]:
%pip install azure-ai-translation-text azure-core python-dotenv --quiet

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: C:\w\repos\foundry-tools\.venv\Scripts\python.exe -m pip install --upgrade pip


In [2]:
import os
from azure.ai.translation.text import TextTranslationClient
from azure.core.credentials import AzureKeyCredential
from azure.core.exceptions import HttpResponseError
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())  # loads .env from repo root

api_key = os.environ["AZURE_TRANSLATOR_KEY"]
region = os.environ["AZURE_TRANSLATOR_REGION"]

credential = AzureKeyCredential(api_key)
client = TextTranslationClient(credential=credential, region=region)
print("Translator client ready.")

Translator client ready.


In [3]:
# Define the text to translate and the target languages
input_text = ["Hello, how are you today?"]
target_languages = ["es", "fr", "de", "it", "ja", "zh-Hans"]

print(f"Input text  : {input_text[0]}")
print(f"Translating to: {', '.join(target_languages)}")

Input text  : Hello, how are you today?
Translating to: es, fr, de, it, ja, zh-Hans


In [4]:
# Translate — source language is auto-detected
try:
    response = client.translate(body=input_text, to_language=target_languages)
    translation_result = response[0] if response else None

    if translation_result:
        detected = translation_result.detected_language
        if detected:
            print(f"Detected source language: {detected.language} (confidence: {detected.score:.2f})\n")

        print("Translations:")
        for t in translation_result.translations:
            print(f"  [{t.to}] {t.text}")

except HttpResponseError as exc:
    if exc.error:
        print(f"Error {exc.error.code}: {exc.error.message}")
    raise

Detected source language: en (confidence: 0.82)

Translations:
  [es] Hola, ¿cómo estás hoy?
  [fr] Bonjour, comment allez-vous aujourd’hui ?
  [de] Hallo wie geht es dir heute?
  [it] Ciao, come stai oggi?
  [ja] こんにちは、今日はお元気ですか?
  [zh-Hans] 你好，今天怎么样？


In [5]:
# Translate multiple sentences at once
multi_input = [
    "The weather is beautiful today.",
    "I love learning new languages.",
    "Azure AI makes translation easy.",
]

try:
    multi_response = client.translate(body=multi_input, to_language=["es", "fr"])

    for i, result in enumerate(multi_response):
        print(f"\nOriginal: {multi_input[i]}")
        for t in result.translations:
            print(f"  [{t.to}] {t.text}")

except HttpResponseError as exc:
    if exc.error:
        print(f"Error {exc.error.code}: {exc.error.message}")
    raise


Original: The weather is beautiful today.
  [es] Hoy hace un tiempo precioso.
  [fr] Le temps est magnifique aujourd’hui.

Original: I love learning new languages.
  [es] Me encanta aprender nuevos idiomas.
  [fr] J’adore apprendre de nouvelles langues.

Original: Azure AI makes translation easy.
  [es] Azure AI facilita la traducción.
  [fr] Azure AI facilite la traduction.
